# tRNA-Arg-TCT-4-1 Dependency Map

Predict the secondary structure of tRNA-Arg-TCT-4-1 (chr1:159141611-159141684, minus strand)
using three different approaches, then compare each against the known cloverleaf contact map.

**Note:** This gene was previously labeled tRNA41/tRNA-Ala but is actually **tRNA-Arg** (anticodon TCT)
per [GtRNAdb](https://gtrnadb.ucsc.edu/genomes/eukaryota/Hsapi38/genes/tRNA-Arg-TCT-4-1.html).

**Sections:**
1. Setup & ground truth
2. **RiNALMo epistasis geometry** — double-mutant embeddings, `epi_R_singles` metric
3. **RiNALMo nucleotide dependency map** — MLM head, unmasked (Da Silva et al. 2025)
4. **RiNALMo embedding perturbation map** — cosine distance on token embeddings, single-pass O(N)
5. Summary comparison + distance normalization
6. **Beyond contacts** — high-scoring pairs NOT in the secondary structure
7. **3D distance matrix** — compare against PDB structure (captures stacking & tertiary contacts)

## 1. Setup & ground truth

In [ ]:
import sys
from pathlib import Path

# Ensure genebeddings is importable (editable install or sys.path fallback)
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "genebeddings").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations, product
from scipy.stats import pearsonr
from seqmat import SeqMat

from genebeddings import VariantEmbeddingDB
from genebeddings.genebeddings import (
    add_epistasis_metrics,
    compute_nucleotide_dependency_map,
    compute_embedding_perturbation_map,
)

In [ ]:
# --- Model config (change these to switch models) ---
from genebeddings.wrappers import RiNALMoWrapper, SpeciesLMWrapper

# Epistasis metrics model
EPI_MODEL_CLS = RiNALMoWrapper
EPI_CONTEXT = 511
EPI_DB_PATH = 'embeddings/rinalmo.db'

# Dependency map model
# RiNALMo is an RNA model — it understands RNA secondary structure.
# DNA-only models (SpeciesLM, NT, etc.) do NOT recover tRNA contacts.
DEPMAP_MODEL_CLS = RiNALMoWrapper
DEPMAP_CONTEXT = 511

# --- tRNA-Arg-TCT-4-1 coordinates ---
CHROM = 'chr1'
START = 159141611
END = 159141684
GENE = 'tRNA41'
CHROM_NUM = '1'
STRAND = 'N'  # negative strand (tRNA is on RC strand)
MAX_DISTANCE = 100
BASES = 'ATGC'
PAD = 256  # flanking context for dependency maps

### Helpers

In [ ]:
BASES = 'ATGC'

def get_all_epistasis_ids(seq, indices, gene, chrom, zyg="N", max_distance=100):
    """Generate epistasis IDs for all position pairs within max_distance, all non-ref ALT combos."""
    items = []
    for i, j in combinations(range(len(seq)), 2):
        if j - i > max_distance:
            continue
        pos1, pos2 = int(indices[i]), int(indices[j])
        ref1, ref2 = seq[i], seq[j]
        if ref1 not in BASES or ref2 not in BASES:
            continue
        alts1 = [b for b in BASES if b != ref1]
        alts2 = [b for b in BASES if b != ref2]
        for alt1, alt2 in product(alts1, alts2):
            id1 = f"{gene}:{chrom}:{pos1}:{ref1}:{alt1}:{zyg}"
            id2 = f"{gene}:{chrom}:{pos2}:{ref2}:{alt2}:{zyg}"
            items.append({'epistasis_id': f"{id1}|{id2}", 'pos1': pos1, 'pos2': pos2})
    return pd.DataFrame(items)


def dotbracket_to_contact_map(ss):
    """Convert dot-bracket notation to a symmetric binary contact matrix."""
    opens = "([{<ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    closes = ")]}>abcdefghijklmnopqrstuvwxyz"
    N = len(ss)
    M = np.zeros((N, N), dtype=int)
    stack = {ch: [] for ch in opens}
    for i, ch in enumerate(ss):
        if ch in opens:
            stack[ch].append(i)
        elif ch in closes:
            j = stack[opens[closes.index(ch)]].pop()
            M[i, j] = M[j, i] = 1
    return M


def upper_tri_flatten(M):
    i, j = np.triu_indices_from(M, k=1)
    return M[i, j]


def zero_diagonal(M):
    """Zero out the diagonal of a matrix."""
    M = M.copy()
    np.fill_diagonal(M, 0)
    return M


def distance_normalize(M):
    """Subtract the mean score at each sequence distance |i-j| to remove proximity bias."""
    M = M.copy()
    N = M.shape[0]
    for d in range(N):
        ii, jj = np.diag_indices(N - d)
        jj = jj + d
        vals = M[ii, jj]
        if len(vals) > 0:
            mu = np.nanmean(vals)
            M[ii, jj] -= mu
            if d > 0:
                M[jj, ii] -= mu
    np.fill_diagonal(M, 0)
    return M


def correlate_with_structure(pred_matrix, M_true, label, flip=False):
    """Score a predicted map against the true contact map.
    
    Zeros out the diagonal of both maps before comparing (upper triangle Pearson r).
    flip=True reverses both axes (use when pred is in forward-strand order but
    ground truth is in RC order).
    """
    pred = pred_matrix.copy()
    if flip:
        pred = pred[::-1, ::-1]
    pred[np.isnan(pred)] = 0
    pred = zero_diagonal(pred)
    truth = zero_diagonal(M_true)
    true_flat = upper_tri_flatten(truth)
    pred_flat = upper_tri_flatten(pred)
    r, p = pearsonr(true_flat, pred_flat)
    print(f"{label}: Pearson r = {r:.4f}, p = {p:.2e}")
    return r, p


def plot_heatmap(matrix, title, cmap='magma', figsize=(10, 8)):
    """Plot a symmetric heatmap."""
    plt.figure(figsize=figsize)
    sns.heatmap(matrix, cmap=cmap, robust=True)
    plt.title(title)
    plt.xlabel("Position")
    plt.ylabel("Position")
    plt.tight_layout()
    plt.show()

### Ground truth: tRNA secondary structure

In [ ]:
# tRNA41 cloverleaf (reverse-complement strand, gaps stripped from alignment)
ss_raw = '.>>>>>>>..>>>>..........<<<<.>>>>>.........................<<<<<................>>>>>.......<<<<<<<<<<<<.'
ss_raw = ss_raw.replace('>', '(').replace('<', ')')
refseq_raw = '.GTCTCTGTGGCGCAATGGAcgA.GCGCGCTGGACTTCTA..................ATCCAGAG...........GtTCCGGGTTCGAGTCCCGGCAGAGATG'
ss = ''.join(c for c, r in zip(ss_raw, refseq_raw) if r != '.')
M_true = dotbracket_to_contact_map(ss)

plt.figure(figsize=(6, 6))
plt.imshow(M_true, cmap='viridis', origin='upper')
plt.title("Ground truth: tRNA41 secondary structure")
plt.xlabel("Position")
plt.ylabel("Position")
plt.colorbar()
plt.show()
print(f"Structure: {len(ss)} nt, {M_true.sum() // 2} base pairs")

# Collect results for final comparison
all_results = []

## 2. RiNALMo: Epistasis geometry (double-mutant embeddings)

For every pair of positions (i, j), embed all 9 double-mutant combinations,
compute epistasis metrics (non-additivity of embedding effects), and
aggregate into a contact-like heatmap.

**Cost:** O(N^2) embeddings &mdash; slow but captures non-linear interactions.

In [ ]:
# Generate variant pairs
s = SeqMat.from_fasta('hg38', CHROM, START, END)
df = get_all_epistasis_ids(s.seq, s.index, GENE, CHROM_NUM, STRAND, MAX_DISTANCE)
print(f"{len(df)} epistasis pairs")

# Run RiNALMo
epi_model = EPI_MODEL_CLS()
db = VariantEmbeddingDB(EPI_DB_PATH)

epi_data = add_epistasis_metrics(
    df, db,
    model=epi_model,
    id_col="epistasis_id",
    context=EPI_CONTEXT,
    show_progress=True,
    force=False,
    batch_size=8,
)

# Plot: epi_R_singles
epi_name = EPI_MODEL_CLS.__name__.replace("Wrapper", "")
dep_singles = epi_data.pivot_table(index='pos1', columns='pos2', values='epi_R_singles', aggfunc='max')
dep_singles = dep_singles.combine_first(dep_singles.T)
plot_heatmap(dep_singles.iloc[::-1, ::-1], f"{epi_name}: epi_R_singles (epistasis geometry)")

# Score — epistasis is indexed by genomic position (forward strand), so flip to RC
r, p = correlate_with_structure(dep_singles.fillna(0).values, M_true, f"{epi_name} epistasis (epi_R_singles)", flip=True)
all_results.append((f"{epi_name} epistasis (epi_R_singles)", r, p))

del epi_model

## 3. Nucleotide dependency map (MLM head, unmasked)

Mutate position i, feed the **full mutated sequence** (no masking), and measure how
nucleotide probabilities change at position j via the MLM head (Da Silva et al. 2025).

**Important:** The tRNA is on the minus strand, so we reverse-complement the sequence
before running the dependency maps. This ensures the model sees the actual RNA.

**Cost:** O(N) forward passes — fast, but requires an MLM model.

In [ ]:
depmap_model = DEPMAP_MODEL_CLS()
depmap_name = DEPMAP_MODEL_CLS.__name__.replace("Wrapper", "")

# Reverse complement: tRNA is on minus strand
s_padded = SeqMat.from_fasta('hg38', CHROM, START - PAD, END + PAD)
s_padded.reverse_complement()
seq_rc_padded = s_padded.seq
tRNA_positions = list(range(PAD, PAD + END - START + 1))

result_nucdep = compute_nucleotide_dependency_map(
    depmap_model, seq_rc_padded,
    positions=tRNA_positions,
    show_progress=True,
)
result_nucdep.plot(title=f"{depmap_name}: nucleotide dependency map (RC strand)")

r, p = correlate_with_structure(result_nucdep.matrix, M_true, f"{depmap_name} nuc-dependency", flip=False)
all_results.append((f"{depmap_name} nuc-dependency", r, p))

## 4. Embedding perturbation map (cosine distance)

Mutate position i, re-embed, measure cosine distance at each token position j.

**Cost:** O(N) forward passes &mdash; fast, works with any model (no MLM head needed).

In [ ]:
result_embed = compute_embedding_perturbation_map(
    depmap_model, seq_rc_padded,
    positions=tRNA_positions,
    diff="cosine",
    show_progress=True,
)
result_embed.plot(title=f"{depmap_name}: embedding perturbation map (RC strand, cosine)")

r, p = correlate_with_structure(result_embed.matrix, M_true, f"{depmap_name} embedding perturbation (cosine)", flip=False)
all_results.append((f"{depmap_name} embedding perturbation (cosine)", r, p))

del depmap_model

## 5. Summary comparison

In [ ]:
df_results = pd.DataFrame(all_results, columns=['Method', 'Pearson r', 'p-value'])
df_results = df_results.sort_values('Pearson r', ascending=False).reset_index(drop=True)
df_results.index += 1
df_results.index.name = 'Rank'
display(df_results)

# Bar chart
fig, ax = plt.subplots(figsize=(10, max(3, len(df_results) * 0.6)))
colors = ['C2' if 'epistasis' in m else 'C1' if 'nuc-dependency' in m else 'C0' for m in df_results['Method']]
ax.barh(range(len(df_results)), df_results['Pearson r'], color=colors)
ax.set_yticks(range(len(df_results)))
ax.set_yticklabels(df_results['Method'], fontsize=10)
ax.set_xlabel('Pearson r (vs tRNA secondary structure)')
ax.set_title('tRNA41 Contact Prediction: Method Comparison')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='C0', label='Embedding perturbation'),
    Patch(color='C1', label='Nucleotide dependency'),
    Patch(color='C2', label='Epistasis geometry'),
], loc='lower right')
plt.tight_layout()
plt.show()

### Distance-normalized comparison

Nearby positions naturally have higher dependency scores regardless of structural
contacts, creating a proximity bias. We subtract the mean score at each sequence
distance |i-j| to remove this confound. This should improve correlations — especially
for epistasis geometry, which is most affected by the distance gradient.

In [ ]:
# Distance-normalize all three maps and re-correlate
embed_dn = distance_normalize(result_embed.matrix.copy())
nucdep_dn = distance_normalize(result_nucdep.matrix.copy())

# Epistasis: flip to RC first, then normalize
epi_raw = dep_singles.fillna(0).values.copy()
epi_raw = epi_raw[::-1, ::-1]
epi_dn = distance_normalize(epi_raw)

dn_results = []

r, p = correlate_with_structure(embed_dn, M_true, f"{depmap_name} embedding perturbation (dist-norm)", flip=False)
dn_results.append((f"{depmap_name} embedding perturbation (dist-norm)", r, p))

r, p = correlate_with_structure(nucdep_dn, M_true, f"{depmap_name} nuc-dependency (dist-norm)", flip=False)
dn_results.append((f"{depmap_name} nuc-dependency (dist-norm)", r, p))

r, p = correlate_with_structure(epi_dn, M_true, f"{epi_name} epistasis (dist-norm)", flip=False)
dn_results.append((f"{epi_name} epistasis (dist-norm)", r, p))

# Combine with original results for comparison
df_dn = pd.DataFrame(all_results + dn_results, columns=['Method', 'Pearson r', 'p-value'])
df_dn = df_dn.sort_values('Pearson r', ascending=False).reset_index(drop=True)
df_dn.index += 1
df_dn.index.name = 'Rank'
display(df_dn)

# Bar chart: original vs distance-normalized
fig, ax = plt.subplots(figsize=(10, max(4, len(df_dn) * 0.5)))
colors = []
for m in df_dn['Method']:
    if 'dist-norm' in m:
        colors.append('#ff7f0e' if 'epistasis' in m else '#2ca02c' if 'nuc-dep' in m else '#1f77b4')
    else:
        colors.append('#ffbb78' if 'epistasis' in m else '#98df8a' if 'nuc-dep' in m else '#aec7e8')
ax.barh(range(len(df_dn)), df_dn['Pearson r'], color=colors)
ax.set_yticks(range(len(df_dn)))
ax.set_yticklabels(df_dn['Method'], fontsize=9)
ax.set_xlabel('Pearson r (vs tRNA secondary structure)')
ax.set_title('Effect of distance normalization on contact prediction')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Figure 1: Three views of tRNA structure from a single foundation model

In [ ]:
# ── Figure 1: Three views of tRNA structure ──────────────────────────────────
from matplotlib.colors import ListedColormap

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'figure.dpi': 300,
})

fig, axes = plt.subplots(2, 2, figsize=(7, 7), constrained_layout=True)

# (a) Ground truth contact map
binary_cmap = ListedColormap(['#f0f0f0', '#2c3e50'])
ax = axes[0, 0]
ax.imshow(M_true, cmap=binary_cmap, origin='upper', aspect='equal', interpolation='none')
ax.set_title('Secondary structure\n(ground truth)')
ax.text(-0.15, 1.05, 'a', transform=ax.transAxes, fontsize=14, fontweight='bold')

# Prepare matrices — all in RC orientation, diagonal zeroed
embed_mat = result_embed.matrix.copy()
embed_mat[np.isnan(embed_mat)] = 0
np.fill_diagonal(embed_mat, 0)

nucdep_mat = result_nucdep.matrix.copy()
nucdep_mat[np.isnan(nucdep_mat)] = 0
np.fill_diagonal(nucdep_mat, 0)

epi_mat = dep_singles.fillna(0).values.copy()
epi_mat = epi_mat[::-1, ::-1]  # flip from genomic coords to RC
np.fill_diagonal(epi_mat, 0)

# Shared color scale across the three predicted maps
all_vals = np.concatenate([embed_mat.ravel(), nucdep_mat.ravel(), epi_mat.ravel()])
vmax = np.percentile(all_vals[all_vals > 0], 99)

# Read r-values from the summary table
r_embed = df_results.loc[df_results['Method'].str.contains('perturbation'), 'Pearson r'].values[0]
r_nucdep = df_results.loc[df_results['Method'].str.contains('nuc-dependency'), 'Pearson r'].values[0]
r_epi = df_results.loc[df_results['Method'].str.contains('epistasis'), 'Pearson r'].values[0]

# (b) Embedding perturbation
ax = axes[0, 1]
im = ax.imshow(embed_mat, cmap='magma', origin='upper', aspect='equal', vmin=0, vmax=vmax)
ax.set_title(f'Embedding perturbation\n(r = {r_embed:.2f} vs contacts)')
ax.text(-0.15, 1.05, 'b', transform=ax.transAxes, fontsize=14, fontweight='bold')

# (c) Nucleotide dependency
ax = axes[1, 0]
ax.imshow(nucdep_mat, cmap='magma', origin='upper', aspect='equal', vmin=0, vmax=vmax)
ax.set_title(f'Nucleotide dependency\n(r = {r_nucdep:.2f} vs contacts)')
ax.text(-0.15, 1.05, 'c', transform=ax.transAxes, fontsize=14, fontweight='bold')

# (d) Epistasis geometry — truncate to match predicted map size
n_pred = embed_mat.shape[0]
ax = axes[1, 1]
ax.imshow(epi_mat[:n_pred, :n_pred], cmap='magma', origin='upper', aspect='equal', vmin=0, vmax=vmax)
ax.set_title(f'Epistasis geometry\n(r = {r_epi:.2f} vs contacts)')
ax.text(-0.15, 1.05, 'd', transform=ax.transAxes, fontsize=14, fontweight='bold')

# Shared colorbar
fig.colorbar(im, ax=axes, shrink=0.6, label='Predicted dependency score')

# Axis labels and ticks
step = 10
n = M_true.shape[0]
for ax in axes.flat:
    ax.set_xlabel('Position')
    ax.set_ylabel('Position')
    ax.set_xticks(range(0, n, step))
    ax.set_yticks(range(0, n, step))

plt.savefig('fig1_three_views.pdf', dpi=300, bbox_inches='tight')
plt.savefig('fig1_three_views.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig1_three_views.pdf and fig1_three_views.png")

### Interpreting the three methods

The three approaches measure fundamentally different things:

**Embedding perturbation (cosine)** — Mutate position i, re-embed the full sequence,
measure how much the embedding at position j changes (cosine distance). This operates
purely in embedding space with no classification head. It asks: *"does the model's
internal representation of position j care about what's at position i?"*

**Nucleotide dependency (Da Silva et al. 2025)** — Mutate position i, feed the full
mutated sequence (no masking), and measure how nucleotide probabilities at position j
change via the MLM head. Unlike log-odds (which masks j), this method preserves full
sequence context. It asks: *"does the model's nucleotide prediction at j change when
I mutate i, given the entire sequence?"*

**Epistasis geometry** — Embed all 4 sequences (WT, single-mut-i, single-mut-j,
double-mut-ij), measure non-additivity of the double mutant effect. It asks: *"is the
combined effect of mutating i and j different from the sum of their individual effects?"*

The perturbation and nucleotide dependency methods measure **direct dependency** (does j
notice i changed?), while epistasis measures **non-linear interaction** (does mutating
both together produce a surprise?). Two positions can be strongly dependent (high
perturbation score) but perfectly additive (low epistasis) — e.g., two bases in the same
stem that both affect stability independently.

For contact prediction, direct dependency works better than non-additivity. And the
embedding space captures richer information than the MLM head output (cosine > nuc-dep),
suggesting the internal representations encode structural relationships that get compressed
when projected down to 4 nucleotide probabilities.

In [ ]:
# Use the best-performing predicted map (embedding perturbation)
# Adjust if a different method scored highest
best_matrix = result_embed.matrix.copy()
# No flip needed — result_embed.matrix is already in RC orientation matching M_true
best_matrix[np.isnan(best_matrix)] = 0
np.fill_diagonal(best_matrix, 0)

# tRNA structural regions (standard numbering, 0-indexed into our 74-nt sequence)
# Reverse-complement strand, so positions are mirrored
N = best_matrix.shape[0]
tRNA_regions = {}
for i in range(N):
    if i < 7:
        tRNA_regions[i] = "acceptor stem"
    elif i < 11:
        tRNA_regions[i] = "D-stem"
    elif i < 25:
        tRNA_regions[i] = "D-loop"
    elif i < 32:
        tRNA_regions[i] = "anticodon stem"
    elif i < 38:
        tRNA_regions[i] = "anticodon loop"
    elif i < 44:
        tRNA_regions[i] = "anticodon stem"
    elif i < 48:
        tRNA_regions[i] = "variable loop"
    elif i < 53:
        tRNA_regions[i] = "T-stem"
    elif i < 61:
        tRNA_regions[i] = "T-loop"
    elif i < 66:
        tRNA_regions[i] = "T-stem"
    else:
        tRNA_regions[i] = "acceptor stem"

# Find high-scoring pairs NOT in the contact map
threshold = np.percentile(best_matrix[best_matrix > 0], 90)  # top 10%
novel_pairs = []
i_idx, j_idx = np.where((best_matrix > threshold) & (M_true == 0))
for i, j in zip(i_idx, j_idx):
    if i < j:  # upper triangle only
        novel_pairs.append({
            'pos_i': i, 'pos_j': j,
            'score': best_matrix[i, j],
            'distance': abs(j - i),
            'region_i': tRNA_regions.get(i, '?'),
            'region_j': tRNA_regions.get(j, '?'),
            'adjacent': abs(j - i) == 1,
        })

df_novel = pd.DataFrame(novel_pairs).sort_values('score', ascending=False)
print(f"High-scoring pairs NOT in contact map (top 10%, score > {threshold:.4f}): {len(df_novel)}")
print(f"\nOf these, {df_novel['adjacent'].sum()} are adjacent (likely stacking)")
print(f"\nTop 20 novel interactions:")
display(df_novel.head(20))

# Annotate cross-region contacts (potential tertiary)
cross_region = df_novel[df_novel['region_i'] != df_novel['region_j']]
cross_region_nonadj = cross_region[~cross_region['adjacent']]
print(f"\nCross-region, non-adjacent pairs (potential tertiary contacts): {len(cross_region_nonadj)}")
display(cross_region_nonadj.head(15))

In [ ]:
# Overlay: predicted map with contact map base pairs marked
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(best_matrix, cmap='magma', origin='upper', aspect='equal')

# Mark known base pairs with green circles
bp_i, bp_j = np.where(M_true == 1)
ax.scatter(bp_j, bp_i, s=30, facecolors='none', edgecolors='lime', linewidths=1.5, label='Known base pairs')

# Mark novel high-scoring pairs with cyan X
if len(df_novel) > 0:
    ax.scatter(df_novel['pos_j'], df_novel['pos_i'], s=20, marker='x',
               color='cyan', linewidths=1, label=f'Novel high-scoring (n={len(df_novel)})')

ax.set_xlabel("Position")
ax.set_ylabel("Position")
ax.set_title("Predicted map with known base pairs (green) and novel interactions (cyan)")
ax.legend(loc='lower left', fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 7. Compare against 3D distance matrix from PDB

The binary contact map only captures Watson-Crick base pairs. A 3D distance matrix
from a crystal structure captures **all** spatial proximity — stacking, tertiary contacts,
backbone geometry.

We use **PDB 1EHZ** (yeast tRNA-Phe, 1.93 A resolution) as the gold-standard free tRNA
structure. All tRNAs share the same L-shaped 3D fold, so this is a valid structural proxy.
We also try **PDB 1F7U** (yeast tRNA-Arg, 2.2 A) for a same-isotype comparison.

We correlate our predicted maps against `1/distance` (closer = stronger interaction).

In [ ]:
import urllib.request
import gzip
import io

def fetch_pdb(pdb_id):
    """Download a PDB file from RCSB."""
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb.gz"
    with urllib.request.urlopen(url) as response:
        return gzip.decompress(response.read()).decode()


def extract_c1prime_coords(pdb_text, chain_id):
    """Extract C1' atom coordinates for each residue in a chain."""
    coords = {}
    for line in pdb_text.splitlines():
        if not line.startswith("ATOM"):
            continue
        atom_name = line[12:16].strip()
        chain = line[21].strip()
        if atom_name != "C1'" or chain != chain_id:
            continue
        resnum = int(line[22:26].strip())
        x = float(line[30:38])
        y = float(line[38:46])
        z = float(line[46:54])
        coords[resnum] = np.array([x, y, z])
    return coords


def coords_to_distance_matrix(coords):
    """Convert residue coords dict to a pairwise distance matrix."""
    resnums = sorted(coords.keys())
    n = len(resnums)
    D = np.zeros((n, n))
    for i, ri in enumerate(resnums):
        for j, rj in enumerate(resnums):
            D[i, j] = np.linalg.norm(coords[ri] - coords[rj])
    return D, resnums


# Fetch structures
PDB_CONFIGS = [
    ('1EHZ', 'A', 'tRNA-Phe (1.93 A, free)'),
    ('1F7U', 'B', 'tRNA-Arg (2.2 A, synthetase complex)'),
]

pdb_distance_matrices = {}
for pdb_id, chain, desc in PDB_CONFIGS:
    print(f"Fetching {pdb_id} chain {chain} ({desc})...")
    pdb_text = fetch_pdb(pdb_id)
    coords = extract_c1prime_coords(pdb_text, chain)
    D, resnums = coords_to_distance_matrix(coords)
    pdb_distance_matrices[pdb_id] = (D, resnums, desc)
    print(f"  {len(resnums)} residues, distance range: {D[D > 0].min():.1f} - {D.max():.1f} A")

In [ ]:
# Compare predicted maps against 3D proximity (1/distance)
# PDB structures may have different lengths than our 74-nt tRNA,
# so we truncate/align to the shorter of the two

predicted_maps = {
    f'{depmap_name} embedding perturbation': result_embed.matrix,
    f'{depmap_name} nuc-dependency': result_nucdep.matrix,
    f'{epi_name} epistasis (epi_R_singles)': dep_singles.fillna(0).values,
}

pdb_results = []
for pdb_id, (D, resnums, desc) in pdb_distance_matrices.items():
    n_pdb = D.shape[0]

    # Convert to proximity: 1/distance (zero on diagonal)
    proximity = np.zeros_like(D)
    mask = D > 0
    proximity[mask] = 1.0 / D[mask]
    np.fill_diagonal(proximity, 0)

    # Plot the 3D proximity map
    plt.figure(figsize=(6, 6))
    plt.imshow(proximity, cmap='magma', origin='upper')
    plt.title(f"PDB {pdb_id}: 3D proximity (1/distance)\n{desc}")
    plt.xlabel("Residue")
    plt.ylabel("Residue")
    plt.colorbar(label="1 / distance (A)")
    plt.show()

    for map_name, pred_raw in predicted_maps.items():
        pred = pred_raw.copy()
        pred[np.isnan(pred)] = 0
        np.fill_diagonal(pred, 0)

        # Align sizes: truncate to min(n_pred, n_pdb)
        n_pred = pred.shape[0]
        n = min(n_pred, n_pdb)
        pred_aligned = pred[:n, :n]
        prox_aligned = proximity[:n, :n]

        true_flat = upper_tri_flatten(prox_aligned)
        pred_flat = upper_tri_flatten(pred_aligned)
        r, p = pearsonr(true_flat, pred_flat)
        print(f"  {map_name} vs {pdb_id} ({desc}): Pearson r = {r:.4f}, p = {p:.2e}")
        pdb_results.append({
            'Method': map_name,
            'PDB': f"{pdb_id} ({desc})",
            'Pearson r': r,
            'p-value': p,
        })

print("\n--- Comparison: binary contact map vs 3D proximity as ground truth ---")
df_pdb = pd.DataFrame(pdb_results).sort_values('Pearson r', ascending=False).reset_index(drop=True)
df_pdb.index += 1
df_pdb.index.name = 'Rank'
display(df_pdb)

## 8. Publication figures

**Figure 2** — Embedding space captures more structural information than the MLM output
layer, and epistasis geometry uniquely captures 3D spatial proximity beyond base pairing.

**Figure 3** — Novel high-scoring pairs predicted by the model are validated as spatially
proximal in the crystal structure, demonstrating that the embedding space encodes
real structural relationships not present in the secondary structure annotation.

In [ ]:
# ── Figure 2: Embedding space vs output space + epistasis for 3D ─────────────
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), constrained_layout=True)

# Colorblind-friendly palette (Okabe-Ito)
C_EMBED = '#0072B2'   # blue
C_NUCDEP = '#D55E00'  # vermillion
C_EPI = '#009E73'     # green
method_labels = ['Embedding\nperturbation', 'Nucleotide\ndependency', 'Epistasis\ngeometry']
method_colors = [C_EMBED, C_NUCDEP, C_EPI]

# (a) vs secondary structure contact map
ax = axes[0]
rs_contact = [r_embed, r_nucdep, r_epi]
bars = ax.barh(range(3), rs_contact, color=method_colors, edgecolor='white', height=0.55)
for i, r in enumerate(rs_contact):
    ax.text(r + 0.02, i, f'{r:.2f}', va='center', fontsize=9, fontweight='bold')
ax.set_yticks(range(3))
ax.set_yticklabels(method_labels, fontsize=9)
ax.set_xlabel('Pearson r')
ax.set_title('vs secondary structure\ncontact map', fontsize=11)
ax.set_xlim(0, 1.0)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.text(-0.3, 1.08, 'a', transform=ax.transAxes, fontsize=14, fontweight='bold')

# (b) vs PDB 3D proximity
ax = axes[1]
# Extract from df_pdb (use 1EHZ)
pdb_1ehz = df_pdb[df_pdb['PDB'].str.contains('1EHZ')].copy()
r_embed_pdb = pdb_1ehz.loc[pdb_1ehz['Method'].str.contains('perturbation'), 'Pearson r'].values[0]
r_nucdep_pdb = pdb_1ehz.loc[pdb_1ehz['Method'].str.contains('nuc-dependency'), 'Pearson r'].values[0]
r_epi_pdb = pdb_1ehz.loc[pdb_1ehz['Method'].str.contains('epistasis'), 'Pearson r'].values[0]
rs_pdb = [r_embed_pdb, r_nucdep_pdb, r_epi_pdb]

bars = ax.barh(range(3), rs_pdb, color=method_colors, edgecolor='white', height=0.55)
for i, r in enumerate(rs_pdb):
    ax.text(r + 0.01, i, f'{r:.2f}', va='center', fontsize=9, fontweight='bold')
ax.set_yticks(range(3))
ax.set_yticklabels(method_labels, fontsize=9)
ax.set_xlabel('Pearson r')
ax.set_title('vs 3D proximity\n(PDB 1EHZ, 1/distance)', fontsize=11)
ax.set_xlim(0, 0.55)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.text(-0.3, 1.08, 'b', transform=ax.transAxes, fontsize=14, fontweight='bold')

# (c) Scatter: embedding perturbation score vs 3D proximity, colored by contact
ax = axes[2]
D_pdb, resnums_pdb, _ = pdb_distance_matrices['1EHZ']
prox_pdb = np.zeros_like(D_pdb)
mask_d = D_pdb > 0
prox_pdb[mask_d] = 1.0 / D_pdb[mask_d]
np.fill_diagonal(prox_pdb, 0)

n = min(embed_mat.shape[0], prox_pdb.shape[0])
pred_flat = upper_tri_flatten(embed_mat[:n, :n])
prox_flat = upper_tri_flatten(prox_pdb[:n, :n])
contact_flat = upper_tri_flatten(M_true[:n, :n])
is_contact = contact_flat.astype(bool)

ax.scatter(pred_flat[~is_contact], prox_flat[~is_contact],
           s=3, alpha=0.25, c='#bdbdbd', label='Non-contact pair', rasterized=True)
ax.scatter(pred_flat[is_contact], prox_flat[is_contact],
           s=28, alpha=0.9, c='#e41a1c', marker='D', label='Known base pair', zorder=5)
ax.set_xlabel('Predicted dependency\n(embedding perturbation)')
ax.set_ylabel(r'3D proximity (1/$d$, PDB 1EHZ)')
ax.set_title('Prediction vs\n3D structure', fontsize=11)
ax.legend(fontsize=8, loc='upper right', framealpha=0.9, markerscale=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.text(-0.2, 1.08, 'c', transform=ax.transAxes, fontsize=14, fontweight='bold')

plt.savefig('fig2_embedding_vs_output.pdf', dpi=300, bbox_inches='tight')
plt.savefig('fig2_embedding_vs_output.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved fig2_embedding_vs_output.pdf and fig2_embedding_vs_output.png")

In [ ]:
# ── Figure 3: Novel contacts validated by 3D structure ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                          gridspec_kw={'width_ratios': [1.2, 0.8]},
                          constrained_layout=True)

# Region color palette
region_colors = {
    'acceptor stem': '#8dd3c7', 'D-stem': '#ffffb3', 'D-loop': '#bebada',
    'anticodon stem': '#fb8072', 'anticodon loop': '#80b1d3',
    'variable loop': '#fdb462', 'T-stem': '#b3de69', 'T-loop': '#fccde5',
}

# (a) Overlay heatmap with known + novel contacts
ax = axes[0]
im = ax.imshow(best_matrix, cmap='magma', origin='upper', aspect='equal')

# Known base pairs
bp_i, bp_j = np.where(M_true == 1)
ax.scatter(bp_j, bp_i, s=35, facecolors='none', edgecolors='#2ca02c',
           linewidths=1.5, label='Known base pairs', zorder=3)

# Novel high-scoring, non-adjacent pairs
novel_nonadj = df_novel[~df_novel['adjacent']]
if len(novel_nonadj) > 0:
    ax.scatter(novel_nonadj['pos_j'], novel_nonadj['pos_i'], s=20, marker='D',
               facecolors='none', edgecolors='#17becf', linewidths=1,
               label=f'Novel predictions (n={len(novel_nonadj)})', zorder=4)

# Region color bands along top and left edges
N_map = best_matrix.shape[0]
for pos in range(N_map):
    region = tRNA_regions.get(pos, '')
    c = region_colors.get(region, '#cccccc')
    ax.plot([pos, pos], [-2.5, -1], color=c, linewidth=3.5, solid_capstyle='butt', clip_on=False)
    ax.plot([-2.5, -1], [pos, pos], color=c, linewidth=3.5, solid_capstyle='butt', clip_on=False)

ax.set_xlabel('Position')
ax.set_ylabel('Position')
ax.set_title('Embedding perturbation map\nwith known and novel contacts')
ax.legend(fontsize=8, loc='lower left', framealpha=0.9)
fig.colorbar(im, ax=ax, shrink=0.7, label='Dependency score')
ax.text(-0.12, 1.05, 'a', transform=ax.transAxes, fontsize=14, fontweight='bold')

# (b) Box plot: 3D proximity by pair category
ax = axes[1]
D_pdb, resnums_pdb, _ = pdb_distance_matrices['1EHZ']
prox_pdb = np.zeros_like(D_pdb)
mask_d = D_pdb > 0
prox_pdb[mask_d] = 1.0 / D_pdb[mask_d]
np.fill_diagonal(prox_pdb, 0)

n = min(best_matrix.shape[0], prox_pdb.shape[0])

# Build a set of novel non-adjacent pair indices for fast lookup
novel_set = set()
for _, row in novel_nonadj.iterrows():
    pi, pj = int(row['pos_i']), int(row['pos_j'])
    if pi < n and pj < n:
        novel_set.add((pi, pj))

# Categorize all upper-triangle pairs
categories, proximities = [], []
i_all, j_all = np.triu_indices(n, k=1)
for idx in range(len(i_all)):
    i, j = int(i_all[idx]), int(j_all[idx])
    prox = prox_pdb[i, j]
    if M_true[i, j] == 1:
        categories.append('Known\nbase pairs')
    elif (i, j) in novel_set:
        categories.append('Novel\npredictions')
    else:
        categories.append('Other\npairs')
    proximities.append(prox)

df_box = pd.DataFrame({'Category': categories, '3D proximity': proximities})
order = ['Known\nbase pairs', 'Novel\npredictions', 'Other\npairs']
palette = {'Known\nbase pairs': '#2ca02c', 'Novel\npredictions': '#17becf', 'Other\npairs': '#bdbdbd'}

sns.boxplot(data=df_box, x='Category', y='3D proximity', order=order,
            palette=palette, ax=ax, fliersize=1.5, width=0.55,
            boxprops=dict(edgecolor='black', linewidth=0.8),
            medianprops=dict(color='black', linewidth=1.2))
ax.set_ylabel(r'3D proximity (1/$d$, PDB 1EHZ)')
ax.set_xlabel('')
ax.set_title('Novel predictions are\nspatially proximal in 3D')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.text(-0.15, 1.05, 'b', transform=ax.transAxes, fontsize=14, fontweight='bold')

# Significance annotation
from scipy.stats import mannwhitneyu
novel_prox = df_box.loc[df_box['Category'] == 'Novel\npredictions', '3D proximity']
other_prox = df_box.loc[df_box['Category'] == 'Other\npairs', '3D proximity']
U, p_mw = mannwhitneyu(novel_prox, other_prox, alternative='greater')
stars = '***' if p_mw < 0.001 else '**' if p_mw < 0.01 else '*' if p_mw < 0.05 else 'n.s.'
y_max = df_box['3D proximity'].quantile(0.98)
ax.plot([1, 1, 2, 2], [y_max*1.02, y_max*1.05, y_max*1.05, y_max*1.02], 'k-', linewidth=0.8)
ax.text(1.5, y_max*1.06, f'{stars}\np = {p_mw:.1e}', ha='center', va='bottom', fontsize=8)

plt.savefig('fig3_novel_contacts.pdf', dpi=300, bbox_inches='tight')
plt.savefig('fig3_novel_contacts.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved fig3_novel_contacts.pdf and fig3_novel_contacts.png")
print(f"Mann-Whitney U test (novel vs other): U={U:.0f}, p={p_mw:.2e}")